<a href="https://colab.research.google.com/github/gnoejh/AIBookGitHub/blob/main/12_grid_world_dp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Grid World Reinforcement Learning

This notebook demonstrates various basic reinforcement learning algorithms applied to a grid world problem. Students will learn by:

1. **Manual Control**: First controlling an agent manually to understand the environment
2. **Dynamic Programming**: Value iteration and policy iteration
3. **Temporal Difference Learning**: Q-learning and SARSA
4. **Monte Carlo Methods**: Learning from complete episodes
5. **Algorithm Comparison**: Comparing different approaches

## Learning Objectives

- Understand the fundamentals of reinforcement learning
- Learn how different algorithms explore and exploit
- Visualize how value functions and policies evolve
- Compare convergence properties of different methods

Let's start by setting up our grid world environment!

In [10]:
# Import required libraries
import numpy as np
from IPython.display import display, clear_output, HTML
import ipywidgets as widgets
import time
from enum import Enum
from typing import Tuple, List, Optional, Dict
import random

print("Libraries imported successfully!")
print("Ready to create our Grid World environment.")
print("Note: Using text-based visualization for better compatibility.")

Libraries imported successfully!
Ready to create our Grid World environment.
Note: Using text-based visualization for better compatibility.


## 1. Grid World Environment

We'll create a customizable grid world where:
- 🤖 **Agent** can move in 4 directions (up, down, left, right)
- 🎯 **Goal** gives positive reward when reached
- 🚫 **Obstacles** block movement
- ⬜ **Empty cells** give small negative reward (encourages efficiency)
- 🔥 **Penalty cells** give large negative reward (pits/traps)

In [11]:
# Define the cell types and actions
class CellType(Enum):
    EMPTY = 0
    OBSTACLE = 1
    GOAL = 2
    PENALTY = 3
    AGENT = 4

class Action(Enum):
    UP = 0
    DOWN = 1
    LEFT = 2
    RIGHT = 3

# GridWorld Environment Class
class GridWorld:
    def __init__(self, width=5, height=5):
        self.width = width
        self.height = height
        self.grid = np.zeros((height, width), dtype=int)
        self.agent_pos = [0, 0]  # [row, col]
        self.goal_pos = [height-1, width-1]
        self.start_pos = [0, 0]
        
        # Rewards
        self.step_reward = -0.1  # Small penalty for each step
        self.goal_reward = 10.0  # Large reward for reaching goal
        self.obstacle_penalty = -1.0  # Penalty for hitting obstacle
        self.penalty_reward = -5.0  # Large penalty for penalty cells
        
        # Initialize grid
        self._setup_default_grid()
        
    def _setup_default_grid(self):
        """Setup a default grid with some obstacles and penalties"""
        # Clear the grid
        self.grid.fill(CellType.EMPTY.value)
        
        # Add some obstacles
        if self.width >= 5 and self.height >= 5:
            self.grid[1, 2] = CellType.OBSTACLE.value
            self.grid[2, 2] = CellType.OBSTACLE.value
            self.grid[3, 1] = CellType.OBSTACLE.value
            
            # Add a penalty cell
            self.grid[2, 3] = CellType.PENALTY.value
        
        # Set goal
        self.grid[self.goal_pos[0], self.goal_pos[1]] = CellType.GOAL.value
        
    def reset(self):
        """Reset the agent to starting position"""
        self.agent_pos = self.start_pos.copy()
        return self.get_state()
    
    def get_state(self):
        """Get current state as tuple (row, col)"""
        return tuple(self.agent_pos)
    
    def is_valid_action(self, action):
        """Check if action is valid from current position"""
        new_pos = self._get_new_position(action)
        return self._is_valid_position(new_pos)
    
    def _get_new_position(self, action):
        """Calculate new position after taking action"""
        row, col = self.agent_pos
        
        if action == Action.UP:
            return [row - 1, col]
        elif action == Action.DOWN:
            return [row + 1, col]
        elif action == Action.LEFT:
            return [row, col - 1]
        elif action == Action.RIGHT:
            return [row, col + 1]
        else:
            return [row, col]  # Invalid action, stay in place
    
    def _is_valid_position(self, pos):
        """Check if position is within bounds and not an obstacle"""
        row, col = pos
        
        # Check bounds
        if row < 0 or row >= self.height or col < 0 or col >= self.width:
            return False
        
        # Check if it's an obstacle
        if self.grid[row, col] == CellType.OBSTACLE.value:
            return False
        
        return True
    
    def step(self, action):
        """Take a step in the environment"""
        # Calculate new position
        new_pos = self._get_new_position(action)
        
        # Check if move is valid
        if not self._is_valid_position(new_pos):
            # Invalid move - stay in place and get penalty
            reward = self.obstacle_penalty
            done = False
        else:
            # Valid move - update position
            self.agent_pos = new_pos
            
            # Calculate reward based on cell type
            cell_type = self.grid[new_pos[0], new_pos[1]]
            
            if cell_type == CellType.GOAL.value:
                reward = self.goal_reward
                done = True
            elif cell_type == CellType.PENALTY.value:
                reward = self.penalty_reward
                done = False
            else:  # Empty cell
                reward = self.step_reward
                done = False
        
        # Return state, reward, done, info
        return self.get_state(), reward, done, {}
    
    def get_possible_actions(self):
        """Get all possible actions from current state"""
        possible_actions = []
        for action in Action:
            if self.is_valid_action(action):
                possible_actions.append(action)
        return possible_actions

print("GridWorld environment created successfully!")
print("✓ Cell types defined (Empty, Obstacle, Goal, Penalty)")
print("✓ Actions defined (Up, Down, Left, Right)")
print("✓ Reward system implemented")

GridWorld environment created successfully!
✓ Cell types defined (Empty, Obstacle, Goal, Penalty)
✓ Actions defined (Up, Down, Left, Right)
✓ Reward system implemented


## 2. Visualization System

Now let's create a visualization system that will show the grid world in real-time. This will help students see exactly what's happening as the agent moves around.

In [12]:
class GridWorldVisualizer:
    def __init__(self, grid_world):
        self.grid_world = grid_world
        
        # Define symbols for different cell types
        self.symbols = {
            CellType.EMPTY.value: '⬜',
            CellType.OBSTACLE.value: '🚫',
            CellType.GOAL.value: '🎯',
            CellType.PENALTY.value: '🔥'
        }
        
        self.agent_symbol = '🤖'
        self.display_widget = widgets.HTML()
        
    def create_grid_html(self):
        """Create HTML representation of the grid"""
        html = "<div style='font-family: monospace; font-size: 24px; line-height: 1.2;'>"
        html += "<h3 style='text-align: center; margin: 10px 0;'>Grid World Environment</h3>"
        html += "<div style='display: inline-block; border: 2px solid #333; padding: 5px;'>"
        
        for i in range(self.grid_world.height):
            html += "<div style='display: flex;'>"
            for j in range(self.grid_world.width):
                # Check if agent is at this position
                if [i, j] == self.grid_world.agent_pos:
                    symbol = self.agent_symbol
                    bg_color = '#E3F2FD'  # Light blue background for agent
                else:
                    cell_type = self.grid_world.grid[i, j]
                    symbol = self.symbols.get(cell_type, '⬜')
                    bg_color = '#F5F5F5'  # Light gray background
                
                html += f"<div style='width: 40px; height: 40px; display: flex; align-items: center; justify-content: center; border: 1px solid #ccc; background-color: {bg_color};'>{symbol}</div>"
            html += "</div>"
        
        html += "</div></div>"
        return html
    
    def update_display(self):
        """Update the complete display"""
        self.display_widget.value = self.create_grid_html()
        
    def add_info_text(self, info_text):
        """Add information text below the grid"""
        current_html = self.display_widget.value
        info_html = f"<div style='text-align: center; margin-top: 10px; padding: 10px; background-color: #f0f0f0; border-radius: 5px;'>{info_text}</div>"
        self.display_widget.value = current_html + info_html
    
    def get_display_widget(self):
        """Get the display widget for showing in the interface"""
        self.update_display()
        return self.display_widget

print("GridWorld visualizer created successfully!")
print("✓ Text-based visualization with emojis")
print("✓ HTML-based grid display")
print("✓ Real-time update capability")

GridWorld visualizer created successfully!
✓ Text-based visualization with emojis
✓ HTML-based grid display
✓ Real-time update capability


## 3. Manual Student Agent Interface

Now let's create an interactive interface where students can manually control the agent and observe the environment step by step. This is the foundation for understanding reinforcement learning!

In [13]:
class ManualAgent:
    def __init__(self, grid_world):
        self.grid_world = grid_world
        self.visualizer = GridWorldVisualizer(grid_world)
        self.total_reward = 0
        self.step_count = 0
        self.episode_history = []
        
        # Create interactive widgets
        self.create_widgets()
        
    def create_widgets(self):
        """Create interactive control widgets"""
        # Action buttons
        self.up_button = widgets.Button(description='↑ UP', button_style='info', layout=widgets.Layout(width='80px'))
        self.down_button = widgets.Button(description='↓ DOWN', button_style='info', layout=widgets.Layout(width='80px'))
        self.left_button = widgets.Button(description='← LEFT', button_style='info', layout=widgets.Layout(width='80px'))
        self.right_button = widgets.Button(description='→ RIGHT', button_style='info', layout=widgets.Layout(width='80px'))
        
        # Control buttons
        self.reset_button = widgets.Button(description='🔄 Reset', button_style='warning', layout=widgets.Layout(width='80px'))
        self.info_button = widgets.Button(description='ℹ️ Info', button_style='success', layout=widgets.Layout(width='80px'))
        
        # Output widget for displaying information
        self.output = widgets.Output()
        
        # Bind button events
        self.up_button.on_click(lambda b: self.take_action(Action.UP))
        self.down_button.on_click(lambda b: self.take_action(Action.DOWN))
        self.left_button.on_click(lambda b: self.take_action(Action.LEFT))
        self.right_button.on_click(lambda b: self.take_action(Action.RIGHT))
        self.reset_button.on_click(lambda b: self.reset_episode())
        self.info_button.on_click(lambda b: self.show_info())
        
        # Layout the buttons
        button_layout = widgets.VBox([
            widgets.HBox([widgets.HTML('<div style="width:80px"></div>'), self.up_button, widgets.HTML('<div style="width:80px"></div>')]),
            widgets.HBox([self.left_button, self.down_button, self.right_button]),
            widgets.HBox([self.reset_button, self.info_button, widgets.HTML('<div style="width:80px"></div>')])
        ])
        
        self.control_panel = widgets.VBox([
            widgets.HTML('<h3>🎮 Manual Agent Controls</h3>'),
            button_layout,
            widgets.HTML('<hr>'),
            self.output
        ])
    
    def take_action(self, action):
        """Execute an action and update display"""
        with self.output:
            clear_output(wait=True)
            
            # Check if action is valid
            if not self.grid_world.is_valid_action(action):
                print(f"❌ Invalid action: {action.name}")
                print(f"Agent cannot move {action.name.lower()} from position {self.grid_world.agent_pos}")
                return
            
            # Take the action
            old_pos = self.grid_world.get_state()
            new_state, reward, done, info = self.grid_world.step(action)
            
            # Update statistics
            self.total_reward += reward
            self.step_count += 1
            self.episode_history.append((old_pos, action.name, reward, new_state))
            
            # Update visualization
            self.visualizer.update_display()
            
            # Display action result
            print(f"🎯 Action: {action.name}")
            print(f"📍 Position: {old_pos} → {new_state}")
            print(f"💰 Reward: {reward:+.1f}")
            print(f"📊 Total Reward: {self.total_reward:+.1f}")
            print(f"👣 Steps: {self.step_count}")
            
            if done:
                print("\n🎉 EPISODE COMPLETE!")
                print(f"🏆 Final Score: {self.total_reward:+.1f} in {self.step_count} steps")
                self.disable_action_buttons()
            else:
                self.show_current_options()
    
    def show_current_options(self):
        """Show available actions from current position"""
        with self.output:
            possible_actions = self.grid_world.get_possible_actions()
            print(f"\n🎯 Available actions from {self.grid_world.get_state()}:")
            for action in possible_actions:
                print(f"  • {action.name}")
    
    def reset_episode(self):
        """Reset the environment for a new episode"""
        with self.output:
            clear_output(wait=True)
            
            # Reset environment and statistics
            self.grid_world.reset()
            self.total_reward = 0
            self.step_count = 0
            self.episode_history = []
            
            # Re-enable buttons
            self.enable_action_buttons()
            
            # Update visualization
            self.visualizer.update_display()
            
            print("🔄 Environment reset!")
            print(f"🤖 Agent position: {self.grid_world.get_state()}")
            print(f"🎯 Goal position: {self.grid_world.goal_pos}")
            self.show_current_options()
    
    def show_info(self):
        """Display helpful information about the environment"""
        with self.output:
            clear_output(wait=True)
            
            print("ℹ️ Grid World Information:")
            print("=" * 30)
            print("🤖 Agent: You control this with arrow buttons")
            print("🎯 Goal: Reach this to complete the episode (+10.0 reward)")
            print("🚫 Obstacles: Cannot move here (-1.0 reward if attempted)")
            print("🔥 Penalty: Dangerous cells (-5.0 reward)")
            print("⬜ Empty: Normal movement (-0.1 reward per step)")
            print("\nReward System:")
            print(f"  • Step penalty: {self.grid_world.step_reward}")
            print(f"  • Goal reward: {self.grid_world.goal_reward}")
            print(f"  • Obstacle penalty: {self.grid_world.obstacle_penalty}")
            print(f"  • Penalty cell: {self.grid_world.penalty_reward}")
            print("\n💡 Try to reach the goal with minimum steps!")
    
    def enable_action_buttons(self):
        """Enable all action buttons"""
        self.up_button.disabled = False
        self.down_button.disabled = False
        self.left_button.disabled = False
        self.right_button.disabled = False
    
    def disable_action_buttons(self):
        """Disable all action buttons"""
        self.up_button.disabled = True
        self.down_button.disabled = True
        self.left_button.disabled = True
        self.right_button.disabled = True
    
    def start_manual_control(self):
        """Start the manual control interface"""
        # Reset and display initial state
        self.reset_episode()
        
        # Display the interface
        return widgets.HBox([
            self.visualizer.get_display_widget(),
            self.control_panel
        ])

print("Manual Agent interface created successfully!")
print("✓ Interactive button controls")
print("✓ Real-time feedback and statistics")
print("✓ Episode management and reset functionality")

Manual Agent interface created successfully!
✓ Interactive button controls
✓ Real-time feedback and statistics
✓ Episode management and reset functionality


## 4. Let's Start! - Manual Control Demo

Now you can experience the grid world yourself! This is the foundation of reinforcement learning - understanding the environment through interaction.

**Instructions:**
1. Run the cell below to create your grid world
2. Use the arrow buttons to move the agent 🤖
3. Try to reach the goal 🎯 with minimum steps
4. Observe how rewards change based on your actions
5. Click 'Reset' to try again with different strategies

In [14]:
# Create the grid world and manual agent
env = GridWorld(width=5, height=5)
manual_agent = ManualAgent(env)

# Start the interactive interface
print("🎮 Manual Control Interface Ready!")
print("Use the buttons to control the agent and learn about the environment.")
print("Click 'Info' button for detailed instructions.")

# Display the interface
interface = manual_agent.start_manual_control()
display(interface)

🎮 Manual Control Interface Ready!
Use the buttons to control the agent and learn about the environment.
Click 'Info' button for detailed instructions.


## 🎯 Learning Questions

After playing with the manual control, think about these questions:

1. **Exploration vs Exploitation**: What's the trade-off between trying new paths vs using known good paths?

2. **Reward Shaping**: How do the small negative step rewards encourage efficient behavior?

3. **State Space**: How many different states (positions) are there in this grid world?

4. **Action Space**: What happens when you try to move into an obstacle or boundary?

5. **Episode Structure**: What defines the end of an episode in this environment?

## 🚀 What's Next?

Now that you understand the environment through manual control, we'll implement various RL algorithms that learn to solve this problem automatically:

- **Value Iteration**: Computing optimal policies through dynamic programming
- **Q-Learning**: Learning from experience without knowing the environment model
- **SARSA**: On-policy learning that's more conservative than Q-learning
- **Monte Carlo**: Learning from complete episodes

Each algorithm will show you different approaches to the same problem!

---

# Part 2: Dynamic Programming Algorithms

Now that you understand the environment through manual control, let's see how algorithms can automatically learn optimal policies! We'll start with **Dynamic Programming** methods that use complete knowledge of the environment.

## 5. Value Iteration Algorithm

### Equations

**Bellman Optimality Equation:**
```
V*(s) = max_a Σ P(s'|s,a) [R(s,a,s') + γ V*(s')]
```

**Value Iteration Update Rule:**
```
V_{k+1}(s) = max_a Σ P(s'|s,a) [R(s,a,s') + γ V_k(s')]
```

**In Our Deterministic Grid World:**
```
V_{k+1}(s) = max_a [R(s,a) + γ V_k(s')]
```

**Policy Extraction:**
```
π*(s) = argmax_a [R(s,a) + γ V*(s')]
```

**Convergence:**
```
max_s |V_{k+1}(s) - V_k(s)| < θ
```

### How Value Iteration Works:
1. **Initialize** all state values to zero
2. **Update** each state's value using the Bellman equation  
3. **Repeat** until values converge
4. **Extract** the optimal policy from the final values

Let's implement this step by step!

In [15]:
class ValueIteration:
    def __init__(self, grid_world, gamma=0.9, theta=0.01):
        self.grid_world = grid_world
        self.gamma = gamma  # Discount factor
        self.theta = theta  # Convergence threshold
        
        # Initialize value function for all states
        self.values = np.zeros((grid_world.height, grid_world.width))
        self.policy = np.full((grid_world.height, grid_world.width), -1, dtype=int)
        
        # Create visualization
        self.visualizer = GridWorldVisualizer(grid_world)
        self.iteration_count = 0
        
    def get_all_states(self):
        """Get all valid states in the grid world"""
        states = []
        for i in range(self.grid_world.height):
            for j in range(self.grid_world.width):
                if self.grid_world.grid[i, j] != CellType.OBSTACLE.value:
                    states.append((i, j))
        return states
    
    def get_state_value(self, state, action):
        """Calculate the value of taking an action from a state"""
        # Temporarily set agent position to calculate next state
        original_pos = self.grid_world.agent_pos.copy()
        self.grid_world.agent_pos = list(state)
        
        # Get the result of taking this action
        next_state, reward, done, _ = self.grid_world.step(action)
        
        # Restore original position
        self.grid_world.agent_pos = original_pos
        
        # Calculate value: reward + gamma * value of next state
        next_value = self.values[next_state[0], next_state[1]]
        value = reward + self.gamma * next_value
        
        return value, next_state
    
    def value_iteration_step(self):
        """Perform one iteration of value iteration"""
        new_values = self.values.copy()
        max_change = 0
        
        # Update value for each state
        for state in self.get_all_states():
            row, col = state
            
            # Skip goal states (terminal states)
            if self.grid_world.grid[row, col] == CellType.GOAL.value:
                continue
                
            # Find the best action for this state
            best_value = float('-inf')
            best_action = None
            
            # Try all possible actions
            for action in Action:
                # Temporarily set agent position
                original_pos = self.grid_world.agent_pos.copy()
                self.grid_world.agent_pos = [row, col]
                
                # Check if action is valid
                if self.grid_world.is_valid_action(action):
                    value, _ = self.get_state_value(state, action)
                    if value > best_value:
                        best_value = value
                        best_action = action
                
                # Restore position
                self.grid_world.agent_pos = original_pos
            
            # Update value and policy
            if best_action is not None:
                new_values[row, col] = best_value
                self.policy[row, col] = best_action.value
                
                # Track maximum change for convergence
                change = abs(new_values[row, col] - self.values[row, col])
                max_change = max(max_change, change)
        
        self.values = new_values
        self.iteration_count += 1
        
        return max_change
    
    def solve(self, max_iterations=100, show_progress=False):
        """Run value iteration until convergence"""
        self.iteration_count = 0
        
        for i in range(max_iterations):
            max_change = self.value_iteration_step()
            
            if show_progress and i % 5 == 0:
                print(f"Iteration {self.iteration_count}: Max change = {max_change:.4f}")
            
            # Check for convergence
            if max_change < self.theta:
                if show_progress:
                    print(f"Converged after {self.iteration_count} iterations!")
                break
        
        return self.values, self.policy
    
    def get_policy_action(self, state):
        """Get the optimal action for a given state"""
        row, col = state
        action_value = self.policy[row, col]
        if action_value >= 0:
            return Action(action_value)
        return None
    
    def create_value_heatmap_html(self):
        """Create HTML heatmap of state values"""
        html = "<div style='font-family: monospace; font-size: 12px; line-height: 1.2;'>"
        html += "<h4 style='text-align: center; margin: 10px 0;'>State Values</h4>"
        html += "<div style='display: inline-block; border: 2px solid #333; padding: 5px;'>"
        
        # Normalize values for color coding
        min_val = np.min(self.values)
        max_val = np.max(self.values)
        val_range = max_val - min_val if max_val != min_val else 1
        
        for i in range(self.grid_world.height):
            html += "<div style='display: flex;'>"
            for j in range(self.grid_world.width):
                # Color based on value
                if self.grid_world.grid[i, j] == CellType.OBSTACLE.value:
                    bg_color = '#000000'
                    text_color = '#ffffff'
                    display_text = '🚫'
                else:
                    # Normalize value to 0-1 range for coloring
                    normalized = (self.values[i, j] - min_val) / val_range
                    # Use green for high values, red for low values
                    if normalized > 0.5:
                        green = 255
                        red = int(255 * (1 - normalized) * 2)
                    else:
                        red = 255
                        green = int(255 * normalized * 2)
                    
                    bg_color = f'rgb({red}, {green}, 100)'
                    text_color = '#000000' if normalized > 0.5 else '#ffffff'
                    display_text = f'{self.values[i, j]:.1f}'
                
                html += f"<div style='width: 60px; height: 40px; display: flex; align-items: center; justify-content: center; border: 1px solid #ccc; background-color: {bg_color}; color: {text_color}; font-weight: bold;'>{display_text}</div>"
            html += "</div>"
        
        html += "</div></div>"
        return html
    
    def create_policy_html(self):
        """Create HTML representation of the policy"""
        action_symbols = {
            Action.UP.value: '↑',
            Action.DOWN.value: '↓',
            Action.LEFT.value: '←',
            Action.RIGHT.value: '→'
        }
        
        html = "<div style='font-family: monospace; font-size: 20px; line-height: 1.2;'>"
        html += "<h4 style='text-align: center; margin: 10px 0;'>Optimal Policy</h4>"
        html += "<div style='display: inline-block; border: 2px solid #333; padding: 5px;'>"
        
        for i in range(self.grid_world.height):
            html += "<div style='display: flex;'>"
            for j in range(self.grid_world.width):
                if self.grid_world.grid[i, j] == CellType.OBSTACLE.value:
                    symbol = '🚫'
                    bg_color = '#000000'
                elif self.grid_world.grid[i, j] == CellType.GOAL.value:
                    symbol = '🎯'
                    bg_color = '#FFD700'
                elif self.grid_world.grid[i, j] == CellType.PENALTY.value:
                    symbol = '🔥'
                    bg_color = '#FF6B6B'
                else:
                    action_val = self.policy[i, j]
                    symbol = action_symbols.get(action_val, '?')
                    bg_color = '#E8F5E8'
                
                html += f"<div style='width: 40px; height: 40px; display: flex; align-items: center; justify-content: center; border: 1px solid #ccc; background-color: {bg_color};'>{symbol}</div>"
            html += "</div>"
        
        html += "</div></div>"
        return html

print("Value Iteration algorithm implemented!")
print("✓ Bellman equation updates")
print("✓ Policy extraction")
print("✓ Value function visualization")
print("✓ Convergence detection")

Value Iteration algorithm implemented!
✓ Bellman equation updates
✓ Policy extraction
✓ Value function visualization
✓ Convergence detection


### Step-by-Step Value Iteration

Want to see exactly how Value Iteration works? Let's create an interactive version where you can run it one step at a time!

In [16]:
class InteractiveValueIteration:
    def __init__(self, grid_world):
        self.vi = ValueIteration(grid_world, gamma=0.9, theta=0.01)
        self.create_widgets()
        
    def create_widgets(self):
        """Create interactive controls for step-by-step execution"""
        self.step_button = widgets.Button(description='➡️ Next Step', button_style='primary')
        self.reset_button = widgets.Button(description='🔄 Reset', button_style='warning')
        self.auto_button = widgets.Button(description='⚡ Auto Run', button_style='success')
        
        self.step_button.on_click(lambda b: self.step())
        self.reset_button.on_click(lambda b: self.reset())
        self.auto_button.on_click(lambda b: self.auto_run())
        
        self.output = widgets.Output()
        self.display_area = widgets.Output()
        
        self.controls = widgets.HBox([self.step_button, self.reset_button, self.auto_button])
        
    def step(self):
        """Execute one iteration step"""
        with self.output:
            clear_output(wait=True)
            
            if self.vi.iteration_count == 0:
                print("🚀 Starting Value Iteration...")
            
            max_change = self.vi.value_iteration_step()
            
            print(f"Iteration {self.vi.iteration_count}:")
            print(f"  Max value change: {max_change:.4f}")
            print(f"  Convergence threshold: {self.vi.theta}")
            
            if max_change < self.vi.theta:
                print("✅ CONVERGED! Algorithm completed.")
                self.step_button.disabled = True
            
            self.update_display()
    
    def reset(self):
        """Reset the algorithm"""
        with self.output:
            clear_output(wait=True)
            
            self.vi = ValueIteration(self.vi.grid_world, gamma=0.9, theta=0.01)
            self.step_button.disabled = False
            
            print("🔄 Value Iteration reset!")
            print("Click 'Next Step' to start the algorithm.")
            
            self.update_display()
    
    def auto_run(self):
        """Run algorithm to completion"""
        with self.output:
            clear_output(wait=True)
            
            print("⚡ Running to completion...")
            values, policy = self.vi.solve(max_iterations=50, show_progress=True)
            print("✅ Completed!")
            
            self.step_button.disabled = True
            self.update_display()
    
    def update_display(self):
        """Update the visualization"""
        with self.display_area:
            clear_output(wait=True)
            
            value_html = self.vi.create_value_heatmap_html()
            policy_html = self.vi.create_policy_html()
            
            value_widget = widgets.HTML(value=value_html)
            policy_widget = widgets.HTML(value=policy_html)
            
            display(widgets.HBox([
                widgets.VBox([
                    widgets.HTML(f"<h4>📈 Values (Iteration {self.vi.iteration_count})</h4>"),
                    value_widget
                ]),
                widgets.VBox([
                    widgets.HTML("<h4>🎯 Current Policy</h4>"),
                    policy_widget
                ])
            ]))
    
    def start_interactive(self):
        """Start the interactive interface"""
        self.reset()
        return widgets.VBox([
            widgets.HTML("<h3>🔬 Interactive Value Iteration</h3>"),
            self.controls,
            self.output,
            self.display_area
        ])

# Create interactive Value Iteration
interactive_vi = InteractiveValueIteration(GridWorld(width=5, height=5))
print("🎮 Interactive Value Iteration ready!")
print("Use the controls to step through the algorithm.")

# Display the interface
display(interactive_vi.start_interactive())

🎮 Interactive Value Iteration ready!
Use the controls to step through the algorithm.


## 6. Policy Iteration Algorithm

### Equations

**Policy Evaluation (Phase 1):**
```
V^π(s) = Σ P(s'|s,π(s)) [R(s,π(s),s') + γ V^π(s')]
```

**In Our Deterministic Grid World:**
```
V^π(s) = R(s,π(s)) + γ V^π(s')
```

**Policy Improvement (Phase 2):**
```
π_{new}(s) = argmax_a [R(s,a) + γ V^π(s')]
```

**Convergence:**
```
π_{new}(s) = π_{old}(s) for all states s
```

### How Policy Iteration Works:
1. **Initialize** with a random policy
2. **Policy Evaluation**: Calculate values for current policy
3. **Policy Improvement**: Update policy based on new values
4. **Repeat** until policy stops changing

Let's implement this!

In [17]:
class PolicyIteration:
    def __init__(self, grid_world, gamma=0.9, theta=0.01):
        self.grid_world = grid_world
        self.gamma = gamma
        self.theta = theta
        
        # Initialize random policy and zero values
        self.policy = np.random.randint(0, 4, (grid_world.height, grid_world.width))
        self.values = np.zeros((grid_world.height, grid_world.width))
        
        # Make sure obstacles and goals have no policy
        for i in range(grid_world.height):
            for j in range(grid_world.width):
                if grid_world.grid[i, j] in [CellType.OBSTACLE.value, CellType.GOAL.value]:
                    self.policy[i, j] = -1
        
        self.iteration_count = 0
        self.evaluation_steps = 0
        
    def get_all_states(self):
        """Get all valid states"""
        states = []
        for i in range(self.grid_world.height):
            for j in range(self.grid_world.width):
                if self.grid_world.grid[i, j] != CellType.OBSTACLE.value:
                    states.append((i, j))
        return states
    
    def policy_evaluation(self, max_iterations=100):
        """Evaluate current policy"""
        self.evaluation_steps = 0
        
        for eval_iter in range(max_iterations):
            new_values = self.values.copy()
            max_change = 0
            
            for state in self.get_all_states():
                row, col = state
                
                # Skip terminal states (goals)
                if self.grid_world.grid[row, col] == CellType.GOAL.value:
                    continue
                
                # Get the action from current policy
                if self.policy[row, col] >= 0:
                    action = Action(self.policy[row, col])
                    
                    # Calculate value for this state under current policy
                    original_pos = self.grid_world.agent_pos.copy()
                    self.grid_world.agent_pos = [row, col]
                    
                    if self.grid_world.is_valid_action(action):
                        next_state, reward, done, _ = self.grid_world.step(action)
                        next_value = self.values[next_state[0], next_state[1]]
                        new_values[row, col] = reward + self.gamma * next_value
                    
                    # Restore position
                    self.grid_world.agent_pos = original_pos
                    
                    # Track change
                    change = abs(new_values[row, col] - self.values[row, col])
                    max_change = max(max_change, change)
            
            self.values = new_values
            self.evaluation_steps += 1
            
            # Check convergence
            if max_change < self.theta:
                break
        
        return self.evaluation_steps
    
    def policy_improvement(self):
        """Improve policy based on current values"""
        policy_stable = True
        
        for state in self.get_all_states():
            row, col = state
            
            # Skip terminal states
            if self.grid_world.grid[row, col] == CellType.GOAL.value:
                continue
            
            old_action = self.policy[row, col]
            
            # Find best action for this state
            best_value = float('-inf')
            best_action = None
            
            for action in Action:
                original_pos = self.grid_world.agent_pos.copy()
                self.grid_world.agent_pos = [row, col]
                
                if self.grid_world.is_valid_action(action):
                    next_state, reward, done, _ = self.grid_world.step(action)
                    value = reward + self.gamma * self.values[next_state[0], next_state[1]]
                    
                    if value > best_value:
                        best_value = value
                        best_action = action
                
                self.grid_world.agent_pos = original_pos
            
            # Update policy
            if best_action is not None:
                self.policy[row, col] = best_action.value
                
                # Check if policy changed
                if old_action != self.policy[row, col]:
                    policy_stable = False
        
        return policy_stable
    
    def solve(self, max_iterations=20, show_progress=False):
        """Run policy iteration until convergence"""
        self.iteration_count = 0
        
        for i in range(max_iterations):
            if show_progress:
                print(f"\n--- Policy Iteration {i+1} ---")
            
            # Policy Evaluation
            eval_steps = self.policy_evaluation()
            if show_progress:
                print(f"Policy evaluation: {eval_steps} steps")
            
            # Policy Improvement
            policy_stable = self.policy_improvement()
            if show_progress:
                print(f"Policy improvement: {'No changes' if policy_stable else 'Policy updated'}")
            
            self.iteration_count += 1
            
            # Check convergence
            if policy_stable:
                if show_progress:
                    print(f"✅ Policy converged after {self.iteration_count} iterations!")
                break
        
        return self.values, self.policy
    
    def create_value_heatmap_html(self):
        """Create HTML heatmap of state values"""
        # Same implementation as ValueIteration
        html = "<div style='font-family: monospace; font-size: 12px; line-height: 1.2;'>"
        html += "<h4 style='text-align: center; margin: 10px 0;'>State Values</h4>"
        html += "<div style='display: inline-block; border: 2px solid #333; padding: 5px;'>"
        
        min_val = np.min(self.values)
        max_val = np.max(self.values)
        val_range = max_val - min_val if max_val != min_val else 1
        
        for i in range(self.grid_world.height):
            html += "<div style='display: flex;'>"
            for j in range(self.grid_world.width):
                if self.grid_world.grid[i, j] == CellType.OBSTACLE.value:
                    bg_color = '#000000'
                    text_color = '#ffffff'
                    display_text = '🚫'
                else:
                    normalized = (self.values[i, j] - min_val) / val_range
                    if normalized > 0.5:
                        green = 255
                        red = int(255 * (1 - normalized) * 2)
                    else:
                        red = 255
                        green = int(255 * normalized * 2)
                    
                    bg_color = f'rgb({red}, {green}, 100)'
                    text_color = '#000000' if normalized > 0.5 else '#ffffff'
                    display_text = f'{self.values[i, j]:.1f}'
                
                html += f"<div style='width: 60px; height: 40px; display: flex; align-items: center; justify-content: center; border: 1px solid #ccc; background-color: {bg_color}; color: {text_color}; font-weight: bold;'>{display_text}</div>"
            html += "</div>"
        
        html += "</div></div>"
        return html
    
    def create_policy_html(self):
        """Create HTML representation of the policy"""
        action_symbols = {
            Action.UP.value: '↑',
            Action.DOWN.value: '↓',
            Action.LEFT.value: '←',
            Action.RIGHT.value: '→'
        }
        
        html = "<div style='font-family: monospace; font-size: 20px; line-height: 1.2;'>"
        html += "<h4 style='text-align: center; margin: 10px 0;'>Policy</h4>"
        html += "<div style='display: inline-block; border: 2px solid #333; padding: 5px;'>"
        
        for i in range(self.grid_world.height):
            html += "<div style='display: flex;'>"
            for j in range(self.grid_world.width):
                if self.grid_world.grid[i, j] == CellType.OBSTACLE.value:
                    symbol = '🚫'
                    bg_color = '#000000'
                elif self.grid_world.grid[i, j] == CellType.GOAL.value:
                    symbol = '🎯'
                    bg_color = '#FFD700'
                elif self.grid_world.grid[i, j] == CellType.PENALTY.value:
                    symbol = '🔥'
                    bg_color = '#FF6B6B'
                else:
                    action_val = self.policy[i, j]
                    symbol = action_symbols.get(action_val, '?')
                    bg_color = '#E8F5E8'
                
                html += f"<div style='width: 40px; height: 40px; display: flex; align-items: center; justify-content: center; border: 1px solid #ccc; background-color: {bg_color};'>{symbol}</div>"
            html += "</div>"
        
        html += "</div></div>"
        return html

print("Policy Iteration algorithm implemented!")
print("✓ Policy evaluation phase")
print("✓ Policy improvement phase")
print("✓ Convergence detection")
print("✓ Visualization support")

Policy Iteration algorithm implemented!
✓ Policy evaluation phase
✓ Policy improvement phase
✓ Convergence detection
✓ Visualization support


### Step-by-Step Policy Iteration

Want to see exactly how Policy Iteration works? Let's create an interactive version where you can run it one step at a time and see both the evaluation and improvement phases!

In [18]:
class InteractivePolicyIteration:
    def __init__(self, grid_world):
        self.pi = PolicyIteration(grid_world, gamma=0.9, theta=0.01)
        self.current_phase = "evaluation"  # "evaluation" or "improvement"
        self.create_widgets()
        
    def create_widgets(self):
        """Create interactive controls for step-by-step execution"""
        self.eval_step_button = widgets.Button(description='📊 Eval Step', button_style='primary')
        self.improve_button = widgets.Button(description='⬆️ Improve', button_style='info')
        self.full_iteration_button = widgets.Button(description='🔄 Full Iteration', button_style='success')
        self.reset_button = widgets.Button(description='🔄 Reset', button_style='warning')
        self.auto_button = widgets.Button(description='⚡ Auto Run', button_style='success')
        
        self.eval_step_button.on_click(lambda b: self.evaluation_step())
        self.improve_button.on_click(lambda b: self.improvement_step())
        self.full_iteration_button.on_click(lambda b: self.full_iteration())
        self.reset_button.on_click(lambda b: self.reset())
        self.auto_button.on_click(lambda b: self.auto_run())
        
        self.output = widgets.Output()
        self.display_area = widgets.Output()
        
        self.controls = widgets.VBox([
            widgets.HTML("<h4>🎛️ Control Phases</h4>"),
            widgets.HBox([self.eval_step_button, self.improve_button]),
            widgets.HTML("<h4>🚀 Quick Actions</h4>"),
            widgets.HBox([self.full_iteration_button, self.reset_button, self.auto_button])
        ])
        
    def evaluation_step(self):
        """Execute one policy evaluation step"""
        with self.output:
            clear_output(wait=True)
            
            if self.pi.iteration_count == 0:
                print("🚀 Starting Policy Iteration...")
                print("📊 Phase 1: Policy Evaluation")
            
            # Store old values to track change
            old_values = self.pi.values.copy()
            
            # Run one evaluation step
            eval_steps = self.pi.policy_evaluation(max_iterations=1)
            
            # Calculate maximum change
            max_change = np.max(np.abs(self.pi.values - old_values))
            
            print(f"📊 Evaluation Step:")
            print(f"  Evaluation steps in this call: {eval_steps}")
            print(f"  Max value change: {max_change:.4f}")
            print(f"  Convergence threshold: {self.pi.theta}")
            
            if max_change < self.pi.theta:
                print("✅ Policy evaluation converged!")
                print("➡️ Ready for policy improvement step")
                self.current_phase = "improvement"
            
            self.update_display()
    
    def improvement_step(self):
        """Execute policy improvement"""
        with self.output:
            clear_output(wait=True)
            
            print("⬆️ Phase 2: Policy Improvement")
            
            # Store old policy
            old_policy = self.pi.policy.copy()
            
            # Run policy improvement
            policy_stable = self.pi.policy_improvement()
            
            # Count policy changes
            policy_changes = np.sum(self.pi.policy != old_policy)
            
            print(f"⬆️ Policy Improvement:")
            print(f"  Policy changes: {policy_changes} states")
            print(f"  Policy stable: {policy_stable}")
            
            if policy_stable:
                print("✅ CONVERGED! Policy is optimal.")
                self.eval_step_button.disabled = True
                self.improve_button.disabled = True
                self.full_iteration_button.disabled = True
            else:
                print("🔄 Starting new iteration...")
                self.pi.iteration_count += 1
                self.current_phase = "evaluation"
            
            self.update_display()
    
    def full_iteration(self):
        """Execute one complete iteration (evaluation + improvement)"""
        with self.output:
            clear_output(wait=True)
            
            if self.pi.iteration_count == 0:
                print("🚀 Starting Policy Iteration...")
            
            print(f"\n🔄 Full Iteration {self.pi.iteration_count + 1}")
            
            # Policy Evaluation
            print("📊 Phase 1: Policy Evaluation")
            eval_steps = self.pi.policy_evaluation()
            print(f"  Converged in {eval_steps} evaluation steps")
            
            # Policy Improvement
            print("⬆️ Phase 2: Policy Improvement")
            old_policy = self.pi.policy.copy()
            policy_stable = self.pi.policy_improvement()
            policy_changes = np.sum(self.pi.policy != old_policy)
            
            print(f"  Policy changes: {policy_changes} states")
            print(f"  Policy stable: {policy_stable}")
            
            self.pi.iteration_count += 1
            
            if policy_stable:
                print("✅ CONVERGED! Algorithm completed.")
                self.eval_step_button.disabled = True
                self.improve_button.disabled = True
                self.full_iteration_button.disabled = True
            
            self.update_display()
    
    def reset(self):
        """Reset the algorithm"""
        with self.output:
            clear_output(wait=True)
            
            self.pi = PolicyIteration(self.pi.grid_world, gamma=0.9, theta=0.01)
            self.current_phase = "evaluation"
            self.eval_step_button.disabled = False
            self.improve_button.disabled = False
            self.full_iteration_button.disabled = False
            
            print("🔄 Policy Iteration reset!")
            print("📊 Click 'Eval Step' to start policy evaluation")
            print("⬆️ Click 'Improve' to run policy improvement")
            print("🔄 Click 'Full Iteration' for complete iteration")
            
            self.update_display()
    
    def auto_run(self):
        """Run algorithm to completion"""
        with self.output:
            clear_output(wait=True)
            
            print("⚡ Running to completion...")
            values, policy = self.pi.solve(max_iterations=20, show_progress=True)
            print("✅ Completed!")
            
            self.eval_step_button.disabled = True
            self.improve_button.disabled = True
            self.full_iteration_button.disabled = True
            self.update_display()
    
    def update_display(self):
        """Update the visualization"""
        with self.display_area:
            clear_output(wait=True)
            
            value_html = self.pi.create_value_heatmap_html()
            policy_html = self.pi.create_policy_html()
            
            value_widget = widgets.HTML(value=value_html)
            policy_widget = widgets.HTML(value=policy_html)
            
            # Show current phase
            phase_text = f"Current Phase: {'📊 Policy Evaluation' if self.current_phase == 'evaluation' else '⬆️ Policy Improvement'}"
            
            display(widgets.VBox([
                widgets.HTML(f"<h4>{phase_text} (Iteration {self.pi.iteration_count})</h4>"),
                widgets.HBox([
                    widgets.VBox([
                        widgets.HTML(f"<h4>📈 Values</h4>"),
                        value_widget
                    ]),
                    widgets.VBox([
                        widgets.HTML("<h4>🎯 Current Policy</h4>"),
                        policy_widget
                    ])
                ])
            ]))
    
    def start_interactive(self):
        """Start the interactive interface"""
        self.reset()
        return widgets.VBox([
            widgets.HTML("<h3>🔬 Interactive Policy Iteration</h3>"),
            widgets.HTML("<p><strong>Policy Iteration</strong> alternates between two phases:</p>"),
            widgets.HTML("<p>📊 <strong>Policy Evaluation:</strong> Calculate values for current policy</p>"),
            widgets.HTML("<p>⬆️ <strong>Policy Improvement:</strong> Update policy based on new values</p>"),
            self.controls,
            self.output,
            self.display_area
        ])

# Create interactive Policy Iteration
interactive_pi = InteractivePolicyIteration(GridWorld(width=5, height=5))
print("🎮 Interactive Policy Iteration ready!")
print("Use the controls to step through the evaluation and improvement phases.")

# Display the interface
display(interactive_pi.start_interactive())

🎮 Interactive Policy Iteration ready!
Use the controls to step through the evaluation and improvement phases.
